In [ ]:
import os
import sys
import json
import warnings
import pickle
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import redirect_stderr

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

# Import the unified configuration
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config import *

from mp_api.client import MPRester
from pymatgen.core import Structure, Composition
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.local_env import CrystalNN
from matminer.featurizers.composition import ElementProperty
from matminer.utils.data import MagpieData

from dotenv import load_dotenv
load_dotenv("../.env", override=True)
MP_API_KEY = os.environ.get("MP_API_KEY")

print(f"Environment initialized! Saving data to: {DATA_DIR}")

In [ ]:
# Fetch Materials 
existing = []
known_ids = set()
if (RAW_DIR / "metadata.json").exists():
    existing = json.loads((RAW_DIR / "metadata.json").read_text()).get("materials", [])
    known_ids = {m["material_id"] for m in existing}

print(f"Loaded {len(existing)} existing materials. Fetching remaining...")
new_mats = []

if MP_API_KEY:
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            band_gap=(BANDGAP_MIN_EV, BANDGAP_MAX_EV),
            num_elements=(1, 6),
            fields=["material_id", "band_gap", "formula_pretty", "elements", "structure"]
        )
        for d in tqdm(docs, desc="Downloading CIFs"):
            mid = str(d.material_id)
            if mid in known_ids or d.band_gap is None or float(d.band_gap) < BANDGAP_MIN_EV:
                continue
            cif_path = CIF_DIR / f"{mid}.cif"
            if not cif_path.exists():
                cif_path.write_text(d.structure.to(fmt="cif"), encoding="utf-8")
            new_mats.append({
                "material_id": mid, "band_gap": float(d.band_gap),
                "formula_pretty": getattr(d, "formula_pretty", None),
                "cif_path": str(cif_path),
            })

all_mats = existing + new_mats
(RAW_DIR / "metadata.json").write_text(json.dumps({"materials": all_mats}, indent=2))
id2bg = {m["material_id"]: float(m["band_gap"]) for m in all_mats}

# XRD Simulation 
GRID = np.arange(TTH_MIN, TTH_MAX + 1e-9, 0.02, dtype=np.float32)
L = len(GRID)

def fwhm_to_sigma(fwhm): return fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))

def _cache_one_xrd(m):
    mid = m["material_id"]
    out = XRD_DIR / f"{mid}.npy"
    if out.exists(): return True
    
    try:
        struct = Structure.from_file(m["cif_path"])
        calc = XRDCalculator(wavelength=WAVELENGTH)
        pattern = calc.get_pattern(struct, two_theta_range=(TTH_MIN, TTH_MAX))
        peak_tth, peak_int = np.asarray(pattern.x, dtype=np.float32), np.asarray(pattern.y, dtype=np.float32)
        
        y = np.zeros(L, dtype=np.float32)
        if peak_tth.size > 0:
            sigma = fwhm_to_sigma(float(FWHM_DEG))
            diff = GRID[:, None] - peak_tth[None, :]
            gauss = np.exp(-0.5 * (diff / sigma) ** 2).astype(np.float32)
            y = gauss @ peak_int
            if y.max() > 0: y = (y / y.max()).astype(np.float32)

        np.save(out, y)
        return True
    except:
        return False

print("Generating/Verifying XRD patterns...")
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
    list(tqdm(ex.map(_cache_one_xrd, all_mats), total=len(all_mats), desc="XRD Patterns"))

materials = [m for m in all_mats if (XRD_DIR / f"{m['material_id']}.npy").exists()]
print(f"Total usable materials: {len(materials)}")

In [ ]:
# Structural Features (Symmetry, Volume, Lattice) 
def _compute_struct_feats(m):
    mid, cif_path = m["material_id"], m["cif_path"]
    symm, volpa, lattice = np.zeros(3, np.float32), np.zeros(1, np.float32), np.zeros(6, np.float32)
    with warnings.catch_warnings(), redirect_stderr(open(os.devnull, "w")):
        warnings.simplefilter("ignore")
        try:
            s = Structure.from_file(str(cif_path))
            volpa = np.array([float(s.lattice.volume) / max(float(len(s)), 1.0)], dtype=np.float32)
            lattice = np.array([s.lattice.a, s.lattice.b, s.lattice.c, s.lattice.alpha, s.lattice.beta, s.lattice.gamma], dtype=np.float32)
            
            sga = SpacegroupAnalyzer(s, symprec=1e-2, angle_tolerance=5.0)
            ops = sga.get_symmetry_operations(cartesian=False)
            is_centro = 1.0 if any(np.all(np.array(op.rotation_matrix, dtype=int) == -np.eye(3, dtype=int)) for op in ops) else 0.0
            symm = np.array([float(sga.get_space_group_number()), float(len(ops)) if ops else 0.0, is_centro], dtype=np.float32)
        except: pass
    return mid, {"symm": symm, "volpa": volpa, "lattice": lattice}

struct_feats = {}
for m in tqdm(materials, desc="Struct Feats"):
    mid, feats = _compute_struct_feats(m)
    struct_feats[mid] = feats

# Geometric Features (CrystalNN) 
def compute_geo_worker(m):
    mid, cif_path = m['material_id'], m['cif_path']
    try:
        struct = Structure.from_file(cif_path)
        cnn = CrystalNN()
        all_cn, all_bond_len, all_ionicity = [], [], []
        
        for i in range(len(struct)):
            info = cnn.get_nn_info(struct, i)
            all_cn.append(len(info))
            for nb in info:
                all_bond_len.append(struct.get_distance(i, nb['site_index']))
                try:
                    all_ionicity.append(abs(struct[i].specie.X - nb['site'].specie.X))
                except: pass

        vec = np.array([
            np.mean(all_cn), np.std(all_cn),
            np.mean(all_bond_len), np.std(all_bond_len), np.min(all_bond_len), np.max(all_bond_len),
            np.mean(all_ionicity) if all_ionicity else 0, np.std(all_ionicity) if all_ionicity else 0, 
            np.min(all_ionicity) if all_ionicity else 0, np.max(all_ionicity) if all_ionicity else 0
        ], dtype=np.float32)
        return mid, np.nan_to_num(vec)
    except:
        return mid, None

geo_raw = {}
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(compute_geo_worker, m) for m in materials]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Geo Feats"):
        mid, res = f.result()
        if res is not None:
            geo_raw[mid] = res

In [ ]:
# Magpie Extraction 
magpie_features = [
    "Number","MendeleevNumber","AtomicWeight","MeltingT","Column","Row",
    "CovalentRadius","Electronegativity",
    "NsValence","NpValence","NdValence","NfValence","NValence",
    "NsUnfilled","NpUnfilled","NdUnfilled","NfUnfilled","NUnfilled",
]
ep = ElementProperty(data_source=MagpieData(), features=magpie_features, stats=["mean", "std_dev", "minimum", "maximum", "range"])

rows = []
for m in tqdm(materials, desc="Formulas -> Compositions"):
    try:
        comp = Composition(m.get("formula_pretty")).reduced_composition
        rows.append((str(m["material_id"]), comp))
    except: pass
    
df_comp = pd.DataFrame(rows, columns=["material_id", "composition"])
df_feat = ep.featurize_dataframe(df_comp, col_id="composition", ignore_errors=True)
df_feat = df_feat.replace([np.inf, -np.inf], np.nan).dropna()
df_feat["material_id"] = df_feat["material_id"].astype(str)

# -Stratified Split
def stratified_split(ids, bins=10):
    rng = np.random.default_rng(SEED)
    bgs = np.array([id2bg[i] for i in ids])
    edges = np.quantile(bgs, np.linspace(0, 1, bins + 1))
    bin_id = np.digitize(bgs, edges[1:-1], right=True)
    
    tr, va, te = [], [], []
    for b in np.unique(bin_id):
        idx = np.where(bin_id == b)[0]
        ids_b = np.array(ids)[idx].copy()
        rng.shuffle(ids_b)
        n_tr, n_va = int(0.8 * len(ids_b)), int(0.1 * len(ids_b))
        tr += ids_b[:n_tr].tolist()
        va += ids_b[n_tr:n_tr+n_va].tolist()
        te += ids_b[n_tr+n_va:].tolist()
    return {"train": tr, "val": va, "test": te}

usable_ids = [m["material_id"] for m in materials if m["material_id"] in id2bg and m["material_id"] in geo_raw]
split = stratified_split(usable_ids)
SPLIT_PATH.write_text(json.dumps(split, indent=2))
print("Split Sizes:", {k: len(v) for k, v in split.items()})

In [ ]:
# Full Feature Scaling (Fitted only on Training Data)
GROUP1_BASE = set(magpie_features[:8])
GROUP2_BASE = set(magpie_features[8:])
def infer_base(col): return next((t for t in reversed(col.split()) if t in magpie_features), None)

cols_g1 = [c for c in df_feat.columns if infer_base(c) in GROUP1_BASE]
cols_g2 = [c for c in df_feat.columns if infer_base(c) in GROUP2_BASE]

magpie_raw = {}
for _, r in df_feat.iterrows():
    x1, x2 = r[cols_g1].to_numpy(dtype=np.float32), r[cols_g2].to_numpy(dtype=np.float32)
    if np.isfinite(x1).all() and np.isfinite(x2).all():
        magpie_raw[str(r["material_id"])] = (x1, x2)

train_mids = [mid for mid in split["train"] if mid in magpie_raw]
X1_tr, X2_tr = np.stack([magpie_raw[m][0] for m in train_mids]), np.stack([magpie_raw[m][1] for m in train_mids])

print("Scaling FULL Magpie features...")
scal_full_g1 = StandardScaler().fit(X1_tr)
scal_full_g2 = StandardScaler().fit(X2_tr)

magpie_all_scaled = {}
for mid, (x1, x2) in magpie_raw.items():
    s1_full = np.clip(scal_full_g1.transform(x1.reshape(1, -1)).flatten(), -5.0, 5.0)
    s2_full = np.clip(scal_full_g2.transform(x2.reshape(1, -1)).flatten(), -5.0, 5.0)
    magpie_all_scaled[mid] = (s1_full, s2_full)

#  Scale Structural Features (including Lattice) 
tr_struct = [m for m in split["train"] if m in struct_feats]
sc_symm = StandardScaler().fit(np.stack([struct_feats[m]["symm"] for m in tr_struct]))
sc_vol = StandardScaler().fit(np.stack([struct_feats[m]["volpa"] for m in tr_struct]))
sc_lat = StandardScaler().fit(np.stack([struct_feats[m]["lattice"] for m in tr_struct]))

struct_scaled = {}
for mid, d in struct_feats.items():
    struct_scaled[mid] = {
        "symm": np.clip(sc_symm.transform(d["symm"].reshape(1, -1)).squeeze(0), -5.0, 5.0),
        "volpa": np.clip(sc_vol.transform(d["volpa"].reshape(1, -1)).squeeze(0), -5.0, 5.0),
        "lattice": np.clip(sc_lat.transform(d["lattice"].reshape(1, -1)).squeeze(0), -5.0, 5.0)
    }

# Scale Geo Features 
tr_geo = [m for m in split["train"] if m in geo_raw]
sc_geo = StandardScaler().fit(np.stack([geo_raw[m] for m in tr_geo]))

geo_scaled = {}
for mid, vec in geo_raw.items():
    geo_scaled[mid] = np.clip(sc_geo.transform(vec.reshape(1, -1)).squeeze(0), -5.0, 5.0)

#Saving the data
print("Saving pre-processed data to disk...")
with open(DATA_DIR / "magpie_all_scaled.pkl", "wb") as f: pickle.dump(magpie_all_scaled, f)
with open(DATA_DIR / "struct_scaled.pkl", "wb") as f: pickle.dump(struct_scaled, f)
with open(DATA_DIR / "geo_scaled.pkl", "wb") as f: pickle.dump(geo_scaled, f)

print(f"Extraction Complete! You are ready to train the models.")